# 🚀 EXAONE 노바코프 챗봇 — RunPod 응용편

`day1/08_노바코프챗봇_EXAONE.ipynb` 에서 만든 챗봇 v0.1 을 RunPod GPU 에서 **실서비스에 가깝게** 다듬어 봅니다.

| STEP | 응용 내용 | 왜 필요한가 |
|------|-----------|-------------|
| 1 | 모듈로 분리된 코드 재사용 (`novacorp.py`) | 노트북 · 터미널 챗봇이 같은 코드를 공유 |
| 2 | 속도 측정 (tokens/sec) | GPU 사양별 체감 성능 비교 |
| 3 | 스트리밍 출력 | 답이 다 만들어질 때까지 기다리지 않기 |
| 4 | 히스토리 트리밍 | 턴이 쌓여도 입력 길이 · 추론 시간 유지 |
| 5 | 자동 평가 + 결과 저장 | 내규 정답 키워드로 정확도 측정, `/workspace/outputs` 에 기록 |
| 6 | (선택) GPT-4o-mini 와 비교 | 로컬 2.4B 모델 vs API 모델 |
| 7 | GPU 메모리 정리 | 다음 실습 전에 VRAM 반납 |

> 먼저 `00_runpod_환경점검.ipynb` 를 통과했는지 확인하세요. 이 노트북은 `runpod/` 폴더에서 실행해야 `novacorp.py` 를 불러올 수 있습니다.

---
## STEP 1. 모델 로드

In [ ]:
import json, os, time
from datetime import datetime
from pathlib import Path

import torch
from novacorp import SYSTEM_PROMPT, generate, load_model, stream, trim_history

print("📥 EXAONE 로드 중... (첫 실행은 다운로드 때문에 2~3분)")
start = time.perf_counter()
tokenizer, model = load_model()
print(f"✅ 로드 완료: {time.perf_counter() - start:.0f}초")
print(f"   VRAM 사용: {torch.cuda.memory_allocated() / 1024**3:.1f} GB")

In [ ]:
# 첫 질문 — generate() 는 응답과 함께 속도 지표를 돌려줍니다
r = generate(tokenizer, model, [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": "연차는 며칠인가요?"},
])
print(r["content"])
print(f"\n입력 {r['input_tokens']} · 출력 {r['output_tokens']} 토큰 | {r['seconds']}초 | {r['tokens_per_sec']} tok/s | {r['finish_reason']}")

---
## STEP 2. 속도 측정 — max_new_tokens 별 tokens/sec

같은 질문을 출력 길이만 바꿔 생성해 봅니다. GPU 를 바꿔 Pod 를 만들었다면 이 표를 비교해 보세요.

In [ ]:
question = [{"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": "노바코프 복리후생을 모두 자세히 설명해 주세요."}]

gpu_name = torch.cuda.get_device_name(0)
print(f"GPU: {gpu_name}\n")
print(f"{'max_new_tokens':>14} | {'출력':>5} | {'시간(s)':>7} | {'tok/s':>6} | 종료")
print("-" * 52)
for max_tok in [64, 128, 256]:
    r = generate(tokenizer, model, question, max_new_tokens=max_tok, temperature=0.0)
    print(f"{max_tok:>14} | {r['output_tokens']:>5} | {r['seconds']:>7} | {r['tokens_per_sec']:>6} | {r['finish_reason']}")

---
## STEP 3. 스트리밍 출력

`generate()` 는 답이 끝날 때까지 아무것도 보여주지 않습니다.
`stream()` 은 `TextIteratorStreamer` 로 **토큰이 생성되는 대로** 출력합니다 — ChatGPT 화면처럼요.

In [ ]:
messages = [{"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": "재택근무 규정을 정리해 주세요."}]

start = time.perf_counter()
first_token_at = None
for piece in stream(tokenizer, model, messages):
    if first_token_at is None and piece:
        first_token_at = time.perf_counter() - start
    print(piece, end="", flush=True)

print(f"\n\n⏱️ 첫 글자까지 {first_token_at:.2f}초 / 전체 {time.perf_counter() - start:.2f}초")

---
## STEP 4. 히스토리 트리밍 — 턴이 쌓여도 느려지지 않게

멀티턴은 매번 **전체 히스토리**를 다시 넣기 때문에 턴이 늘수록 입력 토큰과 추론 시간이 커집니다.
`trim_history(history, max_turns)` 는 system 은 유지하고 최근 N 턴만 남깁니다.

In [ ]:
def count_tokens(messages):
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    return len(tokenizer(prompt, add_special_tokens=False).input_ids)

questions = ["연차는 며칠인가요?", "신청은 언제까지 해야 하나요?", "병가는요?",
             "재택근무는 주 며칠?", "신입도 되나요?", "출장 항공권 규정은?", "방금까지 물어본 걸 요약해줘"]

history = [{"role": "system", "content": SYSTEM_PROMPT}]
print(f"{'턴':>2} | {'전체 히스토리':>11} | {'트리밍(3턴)':>11}")
print("-" * 34)
for i, q in enumerate(questions, 1):
    history.append({"role": "user", "content": q})
    trimmed = trim_history(history, max_turns=3)
    print(f"{i:>2} | {count_tokens(history):>9}토큰 | {count_tokens(trimmed):>9}토큰")
    r = generate(tokenizer, model, trimmed, max_new_tokens=150)
    history.append({"role": "assistant", "content": r["content"]})

print("\n마지막(요약) 답변:\n" + history[-1]["content"])
print("\n💡 트리밍하면 오래된 대화는 잊습니다. 요약 질문이 최근 3턴만 다루는지 확인해 보세요.")

---
## STEP 5. 자동 평가 + 결과 저장

내규에서 정답 키워드를 뽑아 두고, 챗봇 답변에 키워드가 들어 있는지 자동으로 채점합니다.
결과는 Pod 를 꺼도 남도록 `/workspace/outputs` 에 JSONL 로 저장합니다. (Day 2 Judge 평가의 맛보기)

In [ ]:
EVAL_SET = [
    ("입사 1년이 지나면 연차가 며칠 생기나요?",          ["15일"]),
    ("재택근무는 일주일에 최대 며칠인가요?",              ["2일"]),
    ("신입사원도 바로 재택근무 할 수 있나요?",            ["6개월"]),
    ("출장 갈 때 비즈니스석 타도 되나요?",               ["이코노미"]),
    ("비밀번호 규칙이 어떻게 되나요?",                   ["90일", "10자"]),
    ("결혼하면 축하금이 얼마인가요?",                   ["100만"]),
    ("회사 노트북을 잃어버렸어요. 언제까지 신고해야 하나요?", ["24시간"]),
    ("개인 차로 출장 가면 유류비는 얼마인가요?",           ["300원"]),
    ("야간 수당은 얼마인가요?",                         ["명시되어 있지 않"]),   # 내규에 없음 → 안내 문구
]

results = []
for q, keywords in EVAL_SET:
    r = generate(tokenizer, model, [{"role": "system", "content": SYSTEM_PROMPT},
                                    {"role": "user", "content": q}],
                 max_new_tokens=200, temperature=0.0)
    passed = all(k in r["content"] for k in keywords)
    results.append({"question": q, "keywords": keywords, "answer": r["content"],
                    "passed": passed, "seconds": r["seconds"]})
    print(f"{'✅' if passed else '❌'} {q}  ({r['seconds']}s)")

score = sum(x["passed"] for x in results)
print(f"\n정확도: {score}/{len(results)} ({score / len(results) * 100:.0f}%)")

In [ ]:
out_dir = Path("/workspace/outputs")
out_dir.mkdir(parents=True, exist_ok=True)
out_path = out_dir / f"eval_exaone_{datetime.now():%Y%m%d_%H%M%S}.jsonl"

with out_path.open("w", encoding="utf-8") as f:
    for row in results:
        f.write(json.dumps({**row, "model": "EXAONE-3.5-2.4B-Instruct", "gpu": gpu_name}, ensure_ascii=False) + "\n")

print(f"💾 저장: {out_path}")
for row in results:
    if not row["passed"]:
        print(f"\n❌ {row['question']}  (기대 키워드: {row['keywords']})\n   → {row['answer'][:150]}")

---
## STEP 6. (선택) GPT-4o-mini 와 비교

`.env` 에 `OPENAI_API_KEY` 가 있으면 같은 평가셋으로 GPT 를 돌려 비교합니다. 없으면 건너뜁니다.

In [ ]:
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(usecwd=True))

if os.environ.get("OPENAI_API_KEY"):
    from openai import OpenAI
    client = OpenAI()
    gpt_pass = 0
    print(f"{'질문':<34} | EXAONE | GPT")
    print("-" * 52)
    for (q, keywords), ex in zip(EVAL_SET, results):
        resp = client.chat.completions.create(
            model="gpt-4o-mini", temperature=0, max_tokens=200,
            messages=[{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": q}],
        )
        ok = all(k in resp.choices[0].message.content for k in keywords)
        gpt_pass += ok
        print(f"{q[:32]:<34} |   {'✅' if ex['passed'] else '❌'}   | {'✅' if ok else '❌'}")
    print(f"\nEXAONE {score}/{len(EVAL_SET)}  vs  GPT-4o-mini {gpt_pass}/{len(EVAL_SET)}")
else:
    print("OPENAI_API_KEY 없음 → 비교를 건너뜁니다.")

---
## STEP 7. GPU 메모리 정리

다른 노트북에서 모델을 또 올리기 전에 VRAM 을 반납합니다. (커널 재시작도 같은 효과)

In [ ]:
import gc

del model
gc.collect()
torch.cuda.empty_cache()
print(f"VRAM 사용: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")

---
## 🧩 더 해보기

- **터미널 챗봇**: 노트북 없이 SSH 에서 `python runpod/novacorp_chat.py` — 스트리밍 + 대화 기록 저장
- **오래 걸리는 작업**: VS Code 를 닫아도 계속 돌도록 `tmux` 나 `nohup python ... > run.log 2>&1 &` 사용
- **평가셋 늘리기**: `EVAL_SET` 에 내규 항목별 질문을 추가하고 정확도를 GPU · temperature 별로 비교
- **비용 관리**: 실습이 끝나면 Pod 를 **Stop** — 켜져 있는 시간만큼 과금됩니다